In [10]:
from google.colab import files
import os

uploaded = files.upload()

# Get the filename of the uploaded image
for fn in uploaded.keys():
  img_path = fn
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving hsv_cam1.jpg to hsv_cam1.jpg
User uploaded file "hsv_cam1.jpg" with length 49508 bytes


In [12]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
from ipywidgets import interact, IntRangeSlider, IntSlider

# Assuming your image is uploaded as 'photo_bulb.jpg'
img = cv2.imread('hsv_cam1.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

@interact(
    h_range=IntRangeSlider(value=[0, 179], min=0, max=179, step=1, description='Hue Range'),
    s_range=IntRangeSlider(value=[0, 255], min=0, max=255, step=1, description='Sat Range'),
    v_range=IntRangeSlider(value=[200, 255], min=0, max=255, step=1, description='Val Range'),
    closing_size=IntSlider(value=7, min=1, max=31, step=2, description='Core Fill')
)
def update(h_range, s_range, v_range, closing_size):
    lower = np.array([h_range[0], s_range[0], v_range[0]])
    upper = np.array([h_range[1], s_range[1], v_range[1]])

    # 1. Create initial mask
    mask = cv2.inRange(hsv, lower, upper)

    # 2. WHITE CORE FIX: Morphological Closing
    # This connects the colored edges to swallow the white center
    kernel = np.ones((closing_size, closing_size), np.uint8)
    mask_fixed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # 3. Visualization
    result = cv2.bitwise_and(img_rgb, img_rgb, mask=mask_fixed)

    plt.figure(figsize=(15, 5))
    plt.subplot(1, 2, 1); plt.imshow(mask_fixed, cmap='gray'); plt.title("Detection Mask")
    plt.subplot(1, 2, 2); plt.imshow(result); plt.title("Filtered Result")
    plt.show()

    # Print the values to use in your final code
    print(f"Current Config: Lower=[{h_range[0]}, {s_range[0]}, {v_range[0]}], Upper=[{h_range[1]}, {s_range[1]}, {v_range[1]}]")

interactive(children=(IntRangeSlider(value=(0, 179), description='Hue Range', max=179), IntRangeSlider(value=(…

In [ ]:
import cv2
import numpy as np

# Use the values you found in the tuner
LOWER_HSV = np.array([55, 61, 199])
UPPER_HSV = np.array([80, 255, 255])
CORE_FILL_KERNEL = 19

def get_drone_position(frame):
    # 1. Convert to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 2. Thresholding
    mask = cv2.inRange(hsv, LOWER_HSV, UPPER_HSV)

    # 3. White Core Fix (Closing)
    kernel = np.ones((CORE_FILL_KERNEL, CORE_FILL_KERNEL), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # 4. Find the Point
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if contours:
        # Get the largest blob to ignore any leftover tiny distractions
        largest_cnt = max(contours, key=cv2.contourArea)

        # Area filter: Ensure it's not just a tiny speck of noise
        if cv2.contourArea(largest_cnt) > 50:
            M = cv2.moments(largest_cnt)
            if M["m00"] != 0:
                # This is your final localization point
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                return (cx, cy)

    return None

# Test it on your image
test_img = cv2.imread('photo_bulb.jpg')
point = get_drone_position(test_img)
if point:
    print(f"Drone localized at: {point}")

Drone localized at: (593, 465)


In [ ]:
import cv2
import numpy as np

# --- YOUR CALIBRATED HSV VALUES ---
# We are using the values that successfully ignored the white flasher
LOWER_HSV = np.array([14, 46, 200])
UPPER_HSV = np.array([74, 96, 255])
CORE_FILL = 11
MIN_AREA = 20
# ----------------------------------

# 1. Initialize USB Webcam
# '0' is usually the built-in laptop webcam.
# Change to '1' or '2' if you have an external USB webcam plugged in.
cap = cv2.VideoCapture(0)

# Try to set resolution (Webcams usually default to 640x480 anyway)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# NOTE: Most standard webcams do not let OpenCV easily turn off Auto-Exposure.
# If the tracking is jumpy, it's likely because the webcam is auto-adjusting brightness.
cap.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25) # Try to force manual exposure (fails on some webcams)

print("Starting Webcam Tracker... Press 'q' to quit.")

kernel = np.ones((CORE_FILL, CORE_FILL), np.uint8)

while True:
    # 2. Read frame from webcam
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame. Is your webcam being used by another app (like Zoom)?")
        break

    # 3. Processing
    # Standard OpenCV reads webcams in BGR format natively (unlike Picamera2 which uses RGB)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Create the mask
    mask = cv2.inRange(hsv, LOWER_HSV, UPPER_HSV)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # Find contours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    tracking = False
    if contours:
        # Get the largest object
        largest_cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_cnt)

        if area > MIN_AREA:
            tracking = True
            # Draw Bounding Box
            x, y, w, h = cv2.boundingRect(largest_cnt)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Draw Center Point
            cx, cy = x + w//2, y + h//2
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            cv2.putText(frame, f"POS: {cx},{cy}", (x, y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # 4. Display the results in two windows
    # Window 1: What the computer "sees" (Black and white mask)
    cv2.imshow("Detection Mask", mask)

    # Window 2: The final tracked result
    cv2.imshow("Live Tracking", frame)

    # Press 'q' to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Clean up
cap.release()
cv2.destroyAllWindows()

Starting Webcam Tracker... Press 'q' to quit.
Failed to grab frame. Is your webcam being used by another app (like Zoom)?
